# 🏥 Clinical Information Extraction System (Med7-Plus Tutorial)
### An Educational Guide to Clinical NLP, Medication Information Extraction, and EHR Informatics

---

## 📖 1. Introduction: The Clinical EHR Problem
Electronic Health Records (EHR) contain millions of patient records, but **up to 80% of actionable medical information is trapped in unstructured free-text narratives** (progress notes, discharge summaries, nursing reports).

In 2021, Kormilitzin et al. published ***Med7: A transferable clinical natural language processing model for electronic health records*** (*Artificial Intelligence in Medicine*). They introduced a clinical Named-Entity Recognition (NER) pipeline trained on **2 million notes from MIMIC-III** to extract 7 medication attributes:
1. `DRUG` (e.g., Metformin)
2. `STRENGTH` (e.g., 500 mg)
3. `DOSAGE` (e.g., 1 tablet)
4. `ROUTE` (e.g., PO / Oral)
5. `FREQUENCY` (e.g., BID / Twice daily)
6. `FORM` (e.g., Tablet / Solution)
7. `DURATION` (e.g., for 10 days)

### 🚀 What is Med7-Plus?
While the original Med7 identifies isolated spans, modern clinical NLP requires:
- **Relation Linking**: Associating which Strength and Route belongs to which Drug.
- **Context Awareness (n2c2 2022)**: Identifying discontinued or denied medications (Negation).
- **Ontology Mapping**: Normalizing brand/generic names to **RxNorm** CUIs and ATC codes.
- **Interoperability**: Exporting structured prescriptions to **HL7 FHIR R4** standards.

Let's explore each component step-by-step!

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

from med7_plus.extractor import Med7ClinicalExtractor
from med7_plus.evaluator import ClinicalEvaluator
from med7_plus.config import ENTITY_COLORS

print("✅ Med7-Plus modules successfully imported!")

## 🧪 2. Running the Clinical Information Extractor
Let's take a complex clinical discharge summary containing:
- Multiple active medications with different schedules
- A **discontinued** medication due to adverse side effects
- Step-down dosing instructions

In [ ]:
extractor = Med7ClinicalExtractor()

sample_clinical_note = (
    "Patient admitted for acute heart failure exacerbation. "
    "Discharge medications: Furosemide 40 mg tablet PO once daily for 14 days. "
    "Metoprolol succinate 25 mg PO daily. "
    "Discontinued Lisinopril due to dry cough. "
    "Patient advised to take Aspirin 81 mg oral daily with food."
)

result = extractor.extract(sample_clinical_note)

print(f"Detected {len(result['raw_entities'])} raw entities across {len(result['prescriptions'])} medication events.")

### 🔍 Inspecting the Extracted Entities

In [ ]:
for ent in result["raw_entities"]:
    print(f"[{ent['label']:<10}] {ent['text']:<25} (Span: {ent['start']}:{ent['end']})")

## 🔗 3. Medication Slot Linking & Context Extraction
Notice how each drug has its associated attributes linked and its active/discontinued status resolved:

In [ ]:
for p in result["prescriptions"]:
    drug_name = p['drug']['text']
    norm = p['drug']['normalization']
    ctx = p['drug']['context']
    
    print("=" * 60)
    print(f"💊 DRUG: {drug_name} -> RxNorm: {norm.get('generic_name')} (CUI: {norm.get('rxcui')})")
    print(f"   Status:     {'❌ DISCONTINUED / NEGATED' if ctx.get('is_negated') else '✅ ACTIVE'}")
    print(f"   Certainty:  {ctx.get('certainty')}")
    print(f"   Strength:   {p['strength']['text'] if p.get('strength') else 'None'}")
    print(f"   Route:      {p['route']['text'] if p.get('route') else 'None'}")
    print(f"   Frequency:  {p['frequency']['text'] if p.get('frequency') else 'None'}")
    print(f"   Duration:   {p['duration']['text'] if p.get('duration') else 'None'}")

## 📊 4. Clinical Evaluation: Strict vs. Lenient F1 Scoring
In general NLP (CoNLL), exact matching is standard. But in clinical NLP (Kormilitzin et al., 2021; n2c2/i2b2 challenges):
- **Strict Evaluation**: Predicted span `[start, end]` and label must **exactly match** the gold span.
- **Lenient Evaluation**: Predicted span and gold span must **overlap by at least 1 character** and share the same label.

Lenient evaluation is vital clinically because whether a model extracts *"Metoprolol succinate"* or *"Metoprolol"*, the clinical drug concept is still correctly identified!

In [ ]:
# Example evaluation calculation
gold_doc = [
    {"start": 79, "end": 89, "label": "DRUG", "text": "Furosemide"},
    {"start": 90, "end": 95, "label": "STRENGTH", "text": "40 mg"},
    {"start": 130, "end": 150, "label": "DRUG", "text": "Metoprolol succinate"}
]

# Suppose a model predicted slightly different boundaries for Metoprolol
pred_doc = [
    {"start": 79, "end": 89, "label": "DRUG", "text": "Furosemide"},
    {"start": 90, "end": 95, "label": "STRENGTH", "text": "40 mg"},
    {"start": 130, "end": 140, "label": "DRUG", "text": "Metoprolol"} # Boundary difference
]

evaluator = ClinicalEvaluator()
metrics = evaluator.evaluate([gold_doc], [pred_doc])

print(f"Strict  F1: {metrics['micro_strict']['f1']:.4f} (Exact matches only)")
print(f"Lenient F1: {metrics['micro_lenient']['f1']:.4f} (Partial overlaps accepted)")

## 🏥 5. Exporting to HL7 FHIR Interoperability Format
In digital health production environments, extracted medications must be converted into **HL7 FHIR R4 MedicationStatement** resources for ingestion into hospital electronic health record systems (Epic, Cerner, Allscripts).

In [ ]:
import json
fhir_bundle = result["fhir_bundle"]
print(f"Generated FHIR Bundle containing {fhir_bundle['total']} MedicationStatements:")
print(json.dumps(fhir_bundle['entry'][0]['resource'], indent=2))

## 🚀 Summary & Next Steps
- You have experienced end-to-end clinical named entity recognition.
- You understand the difference between isolated span tagging and structured medication slot linking.
- You learned why clinical NLP uses Strict and Lenient F1 evaluation metrics.
- You saw how unstructured narratives become standardized FHIR healthcare payloads.